# centaur-lab data explorer

All connection plumbing lives in the `centaur_db` package — `db.connect()` starts a kubectl port-forward to the in-cluster Postgres if one isn't already up, and tears it down on kernel shutdown.

Use `db.query(conn, sql, params)` for SELECTs (returns a `pandas.DataFrame`) and `db.execute(conn, sql, params)` for INSERT/UPDATE/DELETE (returns rowcount).

In [ ]:
import centaur_db as db

conn = db.connect()
q = lambda sql, params=None: db.query(conn, sql, params)
q("SELECT current_database() AS db, version() AS pg_version").iloc[0].to_dict()

## #local-protocol — recent messages and projected docs

In [ ]:
channel = q(
    "SELECT channel_id, channel_name FROM slack_sync_channels WHERE channel_name = %s",
    ("local-protocol",),
).iloc[0]
channel_id = channel.channel_id

messages = q(
    "SELECT occurred_at, user_id, reply_count, left(text, 240) AS preview "
    "FROM slack_sync_messages WHERE channel_id = %s ORDER BY occurred_at DESC",
    (channel_id,),
)
docs = q(
    "SELECT source_type, title, length(body) AS body_len, source_updated_at "
    "FROM company_context_documents WHERE source = 'slack' AND source_document_id LIKE %s "
    "ORDER BY source_updated_at DESC",
    (f"%{channel_id}%",),
)
print(f"#{channel.channel_name}: {len(messages)} messages, {len(docs)} projected docs")
messages.head(10)

## BM25 search via ParadeDB

Centaur's Postgres is the [ParadeDB](https://docs.paradedb.com) image (`paradedb/paradedb:0.23.0-pg16`), with a `pg_search` BM25 index on `company_context_documents`. `body @@@ 'query'` is the BM25 match operator; `paradedb.score(document_id)` returns the relevance score. This is what the agent's company-context retriever uses under the hood.

In [ ]:
q(
    "SELECT source_type, title, paradedb.score(document_id) AS score, "
    "       left(body, 240) AS preview "
    "FROM company_context_documents "
    "WHERE body @@@ %s "
    "ORDER BY score DESC LIMIT 5",
    ("slack backfill",),
)

## Reply network + PageRank

Build the directed reply graph for this channel — every reply is an edge from `replier → original_poster`, weighted by reply count. Then run PageRank, which is *the same algorithm the channel itself debates* (PPR / SALSA / Eigentrust are all PageRank-family). Node size is proportional to PageRank score: who's the *Eigentrust anchor* of the conversation?

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

edges = q(
    "SELECT reply.user_id AS replier, orig.user_id AS original, count(*) AS weight "
    "FROM slack_sync_messages reply "
    "JOIN slack_sync_messages orig "
    "  ON orig.channel_id = reply.channel_id "
    "  AND orig.message_ts = reply.parent_message_ts "
    "WHERE reply.channel_id = %s "
    "  AND reply.parent_message_ts IS NOT NULL "
    "  AND reply.user_id != orig.user_id "
    "GROUP BY 1, 2",
    (channel_id,),
)

users = q(
    "SELECT user_id, COALESCE(NULLIF(display_name, ''), NULLIF(real_name, ''), substr(user_id, 1, 6)) AS label "
    "FROM slack_sync_users"
)
labels = dict(zip(users.user_id, users.label))

G = nx.DiGraph()
for _, e in edges.iterrows():
    G.add_edge(e.replier, e.original, weight=int(e.weight))
pr = nx.pagerank(G, weight="weight")

fig, ax = plt.subplots(figsize=(11, 8))
pos = nx.spring_layout(G, k=0.6, iterations=60, seed=42)
nx.draw_networkx_edges(
    G, pos,
    alpha=0.35, edge_color="#888", arrows=True, arrowsize=8,
    width=[G[u][v]["weight"] * 0.3 for u, v in G.edges()],
    connectionstyle="arc3,rad=0.08",
)
nx.draw_networkx_nodes(
    G, pos,
    node_size=[pr[n] * 8000 for n in G.nodes()],
    node_color="#4a90e2", alpha=0.85, edgecolors="white", linewidths=1.5,
)
nx.draw_networkx_labels(
    G, pos,
    labels={n: labels.get(n, n[:6]) for n in G.nodes()},
    font_size=9, font_color="#222",
)
ax.set_title(f"#{channel.channel_name} reply network — node size ∝ PageRank", fontsize=13, pad=15)
ax.axis("off")
plt.tight_layout()
plt.show()

pd.DataFrame(
    [
        (labels.get(u, u), pr[u], G.in_degree(u, weight="weight"), G.out_degree(u, weight="weight"))
        for u in pr
    ],
    columns=["author", "pagerank", "replies_received", "replies_sent"],
).sort_values("pagerank", ascending=False).head(10)

## Topic trajectory

For each recurring concept, count messages mentioning it per month (case-insensitive regex on `slack_sync_messages.text`). Stacked area chart shows when each thread of thought entered the discourse and when it peaked — the project's intellectual arc at a glance.

In [ ]:
concepts = {
    "PageRank / PPR / SALSA": r"pagerank|\bppr\b|salsa|page.?rank",
    "Eigentrust":             r"eigentrust|eigen.?trust",
    "ZK proofs":              r"\bzk\b|zero.?knowledge|zkp",
    "Attestation / signing":  r"attestation|signed|signature",
    "Merkle / DAG":           r"merkle|\bdag\b|directed acyclic",
    "Verifiability":          r"verifiab|verification\s+proof|proof of",
}

frames = []
for label, pattern in concepts.items():
    df = q(
        "SELECT date_trunc('month', occurred_at)::date AS month, count(*) AS hits "
        "FROM slack_sync_messages "
        "WHERE channel_id = %s AND text ~* %s "
        "GROUP BY 1 ORDER BY 1",
        (channel_id, pattern),
    )
    df["concept"] = label
    frames.append(df)

traj = (
    pd.concat(frames)
    .pivot_table(index="month", columns="concept", values="hits", fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
traj.plot.area(ax=ax, alpha=0.75, linewidth=0.5)
ax.set_title(
    f"#{channel.channel_name} — concept mentions per month",
    fontsize=12, pad=12,
)
ax.set_xlabel("")
ax.set_ylabel("messages")
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

traj